---
title: "From Structured Data to a Predictive Model"
subtitle: "A workshop on customer churn"
jupyter: data-extraction-lesson
fig-width: 7
fig-height: 4.4
---

A telecommunications company wants to know which customers are likely to leave. Leaving is called **customer churn**. The company already stores a structured record for each customer: contract, services, tenure and charges, together with a label that says whether that customer churned. It also has written communications: complaints, billing messages, support conversations and internal notes.

This workshop is a guided investigation of one question.

**Do the communications contain useful information beyond the structured customer record, once we judge the models on customers they did not see during fitting?**

You will compare three approaches on the same test customers:

1.  **Tabular model.** Use only the structured customer record.
2.  **Combined model.** Use the structured record and numerical summaries of the communications.
3.  **Text-only model.** Use only those numerical summaries.

The model in all three cases is a **logistic regression**: a simple, standard way to turn customer information into a score between 0 and 1. We keep the model the same so that the comparison is about the information, not about a change of algorithm. A logistic regression is a good way to see a complete prediction workflow. It is not automatically the best model for every churn problem.

Three roles stay separate throughout the session.

-   The **computer** calculates scores and performance measures.
-   The **analyst** checks whether the procedure is fair and what the measures mean.
-   The **business** decides whether to act, whom to contact, and at what cost.

```{=html}
<!--
Two hours of practice will help you ask those questions. It will not replace a data scientist.

## Suggested timing

| Stage | What you will do | Guide |
|---|---|---:|
| 1 | Define the business question | 10 min |
| 2 | Prepare the environment and load the data | 10 min |
| 3 | Fit the tabular model | 20 min |
| 4 | Read the first set of results | 15 min |
| 5 | Turn communications into numbers | 20 min |
| 6 | Attach those numbers to the customer table | 10 min |
| 7 | Fit the combined model and compare | 20 min |
| 8 | Fit the text-only model and interpret | 15 min |

The times are a guide. Run the cells from top to bottom. Read the output, then the paragraph under it.
-->
```

::: callout-note
## Optional prompt for an AI assistant

"Explain the difference between training data and test data using the customer churn example."

Use an assistant to understand a command or a result. Check the answer against the tables in this document. An assistant can be wrong. Do not paste real customer communications into an external service unless your organization has authorized that use. The texts in this workshop are synthetic teaching texts.
:::

## Stage 1 — Define the business question

### What churn means

`Churn` is the target. In this dataset it is a text field with two values: `Yes` if the customer left, and `No` if the customer stayed. A **binary** variable has two outcomes. The business question is richer than the label. The company would like a **score for each customer**, somewhere between 0 and 1, that ranks customers by how similar they are to historical customers who left.

A score is not yet a decision. A score of 0.80 does not, by itself, mean "call this customer" or "offer a discount." The company still has to choose an action, a budget and a rule for how high a score must be before someone is contacted. That rule is called a **cutoff**.

Predicting a score is different from predicting only yes or no. The score can be sorted. The company can contact the 200 customers with the highest scores, or it can contact everyone above a cutoff. Those are business choices. The model supplies the ranking.

### What information is available

Two files are used.

**Dataset A, the customer table.** This is the public Telco Customer Churn dataset published on Kaggle: <https://www.kaggle.com/datasets/blastchar/telco-customer-churn>. One row is one customer. The prepared file in this project is `data/raw/telco/telco_customers.csv`. It contains the public fields. The identifier is `customerID`. The target is `Churn`.

The structured fields fall into a few groups:

| Group | Columns |
|------------------------------------|------------------------------------|
| Identifier | `customerID` |
| Who the customer is | `gender`, `SeniorCitizen`, `Partner`, `Dependents` |
| Phone | `tenure`, `PhoneService`, `MultipleLines` |
| Internet and add-on services | `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` |
| Contract and charges | `Contract`, `PaperlessBilling`, `PaymentMethod`, `MonthlyCharges`, `TotalCharges` |
| Outcome | `Churn` |

**Dataset B, the communications.** The file is `data/raw/telco/customer_communications.csv`. One row is one message. A customer can have several messages. The link to the customer table is `customerID`. The same folder contains two much smaller files of rewritten notes. They cover only a handful of customers, so this workshop uses the full file.

The communications were **generated for this course** from the customer records. They are not an export from a live call centre. A complaint about repeated outages *could*, in a real archive, reveal dissatisfaction that no contract field records. Whether *these* texts contain that kind of information is something we will measure, not something we will assume.

The same folder has no date on the messages. We will return to that limitation when we build the text features.

### The question we will answer

On customers the model has not seen, does adding the communications change the quality of the ranking? The text-only model asks a related question: how far do the communications go by themselves?


### A question to write down before any model

If you could contact only 10 percent of the base, whom would you contact first, and which facts would you want in front of you? Keep that sentence. <!--The last stage asks you to compare it with what the models actually used.-->

## Stage 2 — Prepare the Python environment and load the data

### What the environment is

The **Python environment** has to comply with the requirements of the workshop. We will use pandas to hold tables, scikit-learn to fit the model, and matplotlib to draw the charts. An environment keeps those versions together so the same commands can be repeated on another day.

The project instructions already live in `README.md` and `requirements.txt`. Use those. Do not create a second environment file.

### Open a terminal in the project folder

The project folder is the folder that contains `_quarto.yml`, `requirements.txt`, `lessons/` and `data/`. In Cursor or VS Code, open a terminal there. On macOS or Linux:

``` bash
cd /path/to/data-extraction-lesson
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install -r requirements.txt
python -m ipykernel install --user \
  --name data-extraction-lesson \
  --display-name "Python (data-extraction-lesson)"
```

On Windows, activate the environment with `.venv\Scripts\activate` instead of `source .venv/bin/activate`.

If the instructor has already prepared the machines, you only need to activate the existing environment:

``` bash
source .venv/bin/activate
```

Then either run the cells of this document from top to bottom, or render it from the project folder:

``` bash
quarto render lessons/02e-churn-text-ml.qmd
```

The document asks for the kernel named **Python (data-extraction-lesson)**.

### Workshop settings

The next cell finds the project folder, loads the workshop commands, and sets the choices that must stay fixed for a fair comparison.

-   `RANDOM_STATE` is a fixed starting number. It makes the split and the model repeatable.
-   `TEST_SIZE` is the share of customers held out for testing. `0.25` means about one quarter.
-   `N_TEXT_COMPONENTS` is *k*, the number of numerical text columns. You can change it later. You do not need the algebra that produces those columns.
-   `TEXT_MODE` controls the text step. `"RUN"` computes it. `"SAVE"` computes it and stores it. `"LOAD"` reads a stored result and continues. On the machine used to prepare this workshop the text step took a little over two seconds, so the default is `"RUN"`. Use `"LOAD"` if you have already saved a matching result and want to skip the computation. `"LOAD"` fails with an explanation if no matching file exists. It will not invent a result.
-   `DECISION_THRESHOLD` is the cutoff. Scores at or above this value are flagged as predicted churn. The comparison later in the workshop uses `0.5`.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Replace these two paths, the column roles and the two feature lists
# with another tabular file and another text file of the same kind.
TABULAR_FILE = "data/raw/telco/telco_customers.csv"
TEXT_FILE = "data/raw/telco/customer_communications.csv"

if not Path(TABULAR_FILE).exists():
    os.chdir(Path("..").resolve())
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import python.workshop_churn as workshop

workshop.ID_COLUMN = "customerID"
workshop.TARGET_COLUMN = "Churn"
workshop.POSITIVE_LABEL = "Yes"
workshop.TEXT_COLUMN = "text"
workshop.TEXT_ORDER_COLUMN = "text_index"
workshop.TEXT_TYPE_COLUMN = "communication_type"

workshop.NUMERIC_FEATURES = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]
workshop.CATEGORICAL_FEATURES = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
]

workshop.RANDOM_STATE = 42
workshop.TEST_SIZE = 0.25
workshop.N_TEXT_COMPONENTS = 64
workshop.TEXT_MODE = "RUN"
workshop.DECISION_THRESHOLD = 0.5

# Other modes for the text step:
# workshop.TEXT_MODE = "SAVE"  # compute the text columns and save them
# workshop.TEXT_MODE = "LOAD"  # read saved columns; they must match this split and this k
# workshop.TEXT_MODE = "RUN"   # compute now, and do not read or write the saved file

pd.set_option("display.max_columns", 8)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

print("Working directory:", Path.cwd())
workshop.active_settings()

The calculations themselves live in `python/workshop_churn.py`. You can open that file if you want to see the steps. You do not need to edit it.

Confirm that the packages import:

In [ ]:
import sklearn
import matplotlib

print("pandas", pd.__version__)
print("scikit-learn", sklearn.__version__)
print("matplotlib", matplotlib.__version__)

::: callout-note
## Optional prompt

"Explain in simple business terms what this Python command does."

:::

### Load the two datasets

`load_datasets()` reads the two files named in the settings cell. It checks that `Churn` contains only `Yes` and `No`. If a numeric field such as total charge is blank, it replaces that blank with zero and prints how many replacements it made. Zero is a fixed rule, not an average computed from other customers.

In [ ]:
customers, communications = workshop.load_datasets(TABULAR_FILE, TEXT_FILE)
workshop.data_overview(customers, communications)

A row of the customer table is one customer. A row of the communications file is one message. The two files cannot be stacked side by side as they stand: most customers have more than one message, so a direct match would repeat the customer once per message. The model would then treat copies of the same person as if they were different people. Later we will build **one row of text numbers per customer**, and only then attach those numbers.

In [ ]:
customers.head(3)

`head(3)` shows the first three customers.

In [ ]:
communications[[
    workshop.ID_COLUMN,
    workshop.TEXT_ORDER_COLUMN,
    workshop.TEXT_TYPE_COLUMN,
    workshop.TEXT_COLUMN,
]].head(3)

`customerID` is the shared identifier. `text_index` orders one customer's messages. It is not a calendar date. `communication_type` says what kind of message it is. `text` is the message.

In [ ]:
workshop.target_counts(customers)

In [ ]:
workshop.text_type_counts(communications)

<!--

In [ ]:
#| eval: false
workshop.total_charge_check()

The public source file has a few blank total charges. The tenure row in this table tells you who those customers are. The prepared file used in the models already stores those totals as numbers. The loading command reported how many blanks it still found there.
-->

In [ ]:
workshop.timing_summary(communications)

If this table shows no date or time column, the file cannot tell us whether a message was written before the customer left. `text_index` only puts one customer's messages in order. We will not claim that this kind of timing problem has been removed.

## Stage 3 — Build the first logistic regression on the tabular data

### The idea, before the command

Historical customers have characteristics and a known outcome. A logistic regression looks for a weighted combination of those characteristics that separates, as well as a simple weighted sum can, the customers who left from the customers who stayed. For a new customer it turns that weighted sum into a score between 0 and 1.

<!--
We ask the model to pay extra attention to customers who churn, because they are fewer than the customers who stay. The technical name is **balanced class weights**. The practical consequence is important: the score is a ranking signal. After we see the scores, we will compare their average with the actual churn rate before anyone reads a score of 0.70 as "a 70 percent chance."
-->

### One split, used for every model

**Training data** are the customers the model learns from. **Test data** are held out and used only to judge the result. If we judged the model on the customers it learned from, we would measure memory as well as prediction.

The split is **stratified** on `Churn`: the share of churners is kept similar in both groups. 
<!--
It is done once, at customer level, and then reused. Splitting messages instead of customers could put one person in both groups.
-->

`customerID` is not a predictor. It names a person, and it would let a model memorize individuals rather than learn a pattern that applies to someone new. `Churn` is not a predictor. It is the answer.

Numeric fields are `tenure`, `MonthlyCharges` and `TotalCharges`. They are rescaled using the training customers only, then the same rescaling is applied to the test customers. A charge of 70 and a tenure of 70 are different facts; rescaling stops a field with large units from dominating for that reason alone.

Fields such as contract and payment method are categories. Each category (present in the training data) becomes a set of yes/no columns. The technical name is **one-hot encoding**. 

<!--
A contract with three levels becomes three yes/no columns. The list of levels is taken from the training customers. An unseen level in the test set would be recorded as all zeros rather than stopping the workshop.
-->

`TotalCharges` and `tenure` describe related facts about the same history. The model may use both. That does not make each one a separate cause of churn.

In [ ]:
split = workshop.prepare_train_test(customers)
workshop.split_summary(split)

### Fit the tabular model

`run_tabular_model()` fits the logistic regression on the training customers and scores both groups. The cutoff is the value in `DECISION_THRESHOLD`.

In [ ]:
tabular = workshop.run_tabular_model(split)
workshop.show_metrics(tabular)

The table has a training row and a test row. Accuracy, precision, recall and F1 use the cutoff. **ROC-AUC** does not. It summarizes the ranking across cutoffs. We will define each measure in the next stage, using the test row as the one that speaks about prediction.

<!--
::: callout-note
## Optional prompt

"Explain in simple business terms what this Python command does: `tabular = workshop.run_tabular_model(split)`."
:::
-->


## Stage 4 — Understand and evaluate the first model

### A rule that ignores the customer record

Most customers stay. A rule that predicts "stay" for everyone will therefore be right quite often, while finding none of the customers who leave. That rule is the reference for accuracy.

In [ ]:
always_stay = workshop.baseline_always_negative(split)
workshop.show_metrics(always_stay)

In [ ]:
workshop.explain_accuracy(tabular, always_stay)

**Accuracy** is the share of test customers whose yes/no flag matches what actually happened. **Recall** is the share of churners the model flags. **Precision** is the share of flagged customers who really churned. **F1** is a single compromise between precision and recall. It is convenient, and it hides the trade-off, so we still read precision and recall separately.

No one of these numbers describes the model.

### Who was flagged

A **confusion matrix** crosses the actual outcome with the flag. The plain-language counts below are the test customers. In technical shorthand, flagged churners are true positives, flagged stayers are false positives, missed churners are false negatives, and unflagged stayers are true negatives.

In [ ]:
workshop.confusion_table(tabular)

In [ ]:
workshop.plot_confusion(tabular)

In [ ]:
workshop.explain_confusion(tabular)

If the company contacts every flagged customer, recall is the share of leavers it will find, and precision is the share of those contacts who were going to leave. The rest of the contacts are false alarms. Which of those is more expensive is a business question. The model does not answer it.

### Ranking, across cutoffs

**ROC-AUC** measures how often a randomly chosen churner receives a higher score than a randomly chosen stayer. A value of 0.5 means the ranking is no better than chance. A value of 1 means every churner is ranked above every stayer. The curve shows that trade-off as the cutoff moves: the horizontal axis is the share of stayers who are flagged, and the vertical axis is the share of churners who are flagged.

In [ ]:
workshop.plot_roc([tabular])

In [ ]:
workshop.explain_train_test(tabular)

Use the test ROC-AUC when you talk about customers the model had not seen. One split is one careful experiment. A data scientist would repeat the split, or use another validation design, before a deployment decision. We do not search for settings that look best on the test customers. Doing that would spend the test set during fitting, and the comparison would no longer be honest.

### The scores themselves

In [ ]:
workshop.plot_probabilities(tabular)

In [ ]:
workshop.explain_scores(tabular)

The dashed line is the cutoff. Moving it changes who is flagged. It does not refit the model.

In [ ]:
tabular_lower_cutoff = workshop.apply_threshold(tabular, 0.30)
workshop.show_metrics(tabular_lower_cutoff)

In [ ]:
workshop.explain_threshold_change(tabular, tabular_lower_cutoff)

You can change `0.30` and run those two cells again. Leave `DECISION_THRESHOLD` at `0.5` for the comparison of the three models, so they are judged with the same rule.

::: callout-note
## Optional prompt

"What does ROC-AUC tell me about this model, and what does it not tell me?"
:::

## Stage 5 — Turn communications into numerical features

### Why the model cannot read the notes directly

A logistic regression adds weighted fields. It cannot add the sentence "the line has been unreliable" to a monthly charge. We first turn each customer's messages into numbers.

The existing procedure has three parts.

1.  **One document per customer.** The customer's messages are joined in `text_index` order. A customer with no messages would keep an empty document and would stay in the table.
2.  **TF-IDF**, short for term frequency–inverse document frequency. Each word, and each two-word phrase, receives a score. The score is high when the expression is common in this customer's document and uncommon across the other training documents. Very common English words are set aside. 
<!--
An expression that appears in only one training document is set aside as well. 
-->
The training step keeps up to 5,000 expressions.
3.  **LSA**, latent semantic analysis. A compression method based on **singular value decomposition (SVD)** reduces those thousands of scores to *k* numbers, named `text_1`, `text_2`, and so on, up to `text_k`. Customers who use related wording end up with similar numbers. You do not need SVD to use the result. You do need to know that *k* is `N_TEXT_COMPONENTS` in the settings cell.

These columns are a numerical summary of patterns in the writing. They are not complaint counts, and they do not have business names such as "billing anger."

The word list and the compression are learned from the **training customers only**. The same learned step is then applied to the test customers. If the test customers were allowed to influence the word list, information from the test set would leak into the model and the evaluation would look better than it is.

```{mermaid}
flowchart LR
  A[Several messages per customer] --> B[One document per customer]
  B --> C[Word scores, TF-IDF]
  C --> D["text_1 to text_k, LSA"]
```

### A three-line illustration

The next cell is a teaching example. The three lines are not customer records, and they are not fed into the churn models. It uses every word because the example is tiny. The real step uses the restrictions described above, and it uses 64 columns rather than two.

In [ ]:
illustration = workshop.illustrate_text_encoding()
illustration["word_scores"]

In [ ]:
illustration["compressed"]

The word-score table is wide enough to read in full. The compressed table is the same idea as `text_1` and `text_2` in the real pipeline: several word scores become a few numbers. The similarity printed above the table tells you which teaching lines use the more overlapping wording.

### Run the text step

This cell uses `TEXT_MODE` and `N_TEXT_COMPONENTS` from the settings cell. `"RUN"` computes the columns. `"SAVE"` computes them and writes `data/processed/text_model_cache/`. `"LOAD"` reads a saved file only when the split, the text file, the number of columns and the processing rules match. A mismatch stops with an instruction. It does not substitute an older or unrelated file.

To recompute and store a new file, set `workshop.TEXT_MODE = "SAVE"` in the settings cell and run again from that cell downward. To force a fresh computation without reading the saved file, set `workshop.TEXT_MODE = "RUN"`.

In [ ]:
text_features = workshop.build_text_features(split, communications)
workshop.text_feature_summary(text_features)

In [ ]:
workshop.preview_text_features(text_features)

The preview shows `customerID` and the first text columns. The summary reports how many training documents were used to fit TF-IDF and LSA, and it must report that zero test documents were used for that fit. "Share of training word-score variation retained" says how much of the variation in the word scores the *k* columns keep. The rest is discarded as smaller detail. That percentage is a description of the compression, not a measure of churn prediction.

::: callout-note
## Optional prompt

"Help me understand why adding customer communications might improve a churn prediction model. Also explain how a text file could make the model look better for a reason a manager should not trust."
:::

## Stage 6 — Combine the structured fields and the text columns

The text columns can now be treated as ordinary numeric predictors. `combine_with_text()` attaches them to the training customers and the test customers by `customerID`.

The checks underneath the command are the ones that protect the comparison.

-   The number of rows after the match must equal the number before it. A larger number would mean a duplicated customer. A smaller number would mean a lost customer.
-   A customer with no text stays in the table. The text columns are filled with zeros. Zero here means "no document was available." It is not a neutral topic score. In the file used today, check whether that count is zero.
-   `customerID` is the match key only. It is not given to the logistic regression as a predictor.
-   `Churn` stays the outcome. It is not copied into the predictors.

The combined table holds two kinds of information: the structured account fields, and a numerical summary of the free-text messages. They enter the model as one list of columns. The technical name for joining them before the model is **early fusion**. The idea is simply that the model sees both blocks at once.

In [ ]:
combined_data = workshop.combine_with_text(split, text_features)
combined_data["check"]

## Stage 7 — Fit and evaluate the combined model

The command below fits a second logistic regression. The model family, the cutoff, the training customers and the test customers are the same as in Stage 3. The new ingredient is the block of text columns. Those columns were produced by a text step fitted on the training customers only.

In [ ]:
combined = workshop.run_combined_model(combined_data)
workshop.show_metrics(combined)

In [ ]:
workshop.explain_train_test(combined)

In [ ]:
workshop.plot_probabilities(combined)

In [ ]:
workshop.explain_score_shape(combined)

Compare this histogram with the tabular one. Scores that sit between the two ends mean the cutoff still changes who would be contacted. A pile of scores at 0 and at 1 would mean the model is very sure about most customers.

### The tabular model and the combined model on the same test customers

In [ ]:
workshop.compare_models([tabular, combined])

In [ ]:
workshop.plot_roc([tabular, combined])

Read the test ROC-AUC column as the ranking comparison. Read recall and precision as the comparison at the shared cutoff of 0.5. A gain in ranking is one fact. A gain large enough to pay for collecting and processing communications is a second fact. A reason for the business to change a retention process is a third fact. The table can support the first. It does not settle the second or the third.

We still have to fit the text-only model before drawing the full comparison. The checks below confirm that these two models used the same test customers and that the text step did not learn from them.

In [ ]:
workshop.evaluation_checks(split, [tabular, combined], text_features)

## Stage 8 — Fit and evaluate the text-only model

The third model uses the text columns and no structured fields. Same training customers, same test customers, same cutoff, same kind of logistic regression.

In [ ]:
text_only = workshop.run_text_only_model(combined_data)
workshop.show_metrics(text_only)

### The three models together

In [ ]:
workshop.compare_models([tabular, combined, text_only])

In [ ]:
workshop.plot_metric_bars([tabular, combined, text_only])

In [ ]:
workshop.plot_roc([tabular, combined, text_only])

In [ ]:
workshop.evaluation_checks(split, [tabular, combined, text_only], text_features)

In [ ]:
workshop.explain_model_comparison([tabular, combined, text_only])

::: callout-important
If a test ROC-AUC sits at the top of the scale, treat that as a warning before you take the number to a meeting. Text alone should not rank the customers perfectly. A gain for the combined model, if the table shows one, can be a limited addition to the structured fields.
:::

Several outcomes were possible, and a manager should be able to name them.

-   Text alone can contain information about dissatisfaction that the structured fields miss.
-   The structured fields can contain predictive information that the messages never state.
-   Using both can help when the two sources are complementary.
-   The combined model can also fail to beat the simpler ones. Nothing in the method guarantees an improvement.


<!--
On **this** file, use the table and the sentence above, not a prior belief about text.

The comparison is not a causal finding. A higher score does not show that a contract, a price or a complaint made the customer leave. It shows how well a score, built from the fields we allowed, ranks historical churners above historical stayers on this test set.
-->

### All of the test results on one page

In [ ]:
workshop.compare_models([always_stay, tabular, combined, text_only])

### What a manager should ask before anyone uses a score

Return to the sentence you wrote in Stage 1. Then ask at least these questions.

1.  Does the model improve performance on customers it did not see during fitting? The test rows are the evidence in this workshop. One split still leaves uncertainty.
2.  Is any improvement large enough to pay for collecting, storing and processing the communications?
3.  Are the scores reliable enough for the decision, or are they only a ranking? The average score and the cutoff discussion in Stage 4 are the start of that question, not the end of it.
4.  What action follows a high score? A call, an offer, a service fix and a decision to do nothing are different actions with different costs.
5.  Were the notes available before the outcome? This file has no message date, so it cannot answer that.
6.  Did any field, including the wording of a note, already contain the outcome?
7.  What has to be settled on data quality, privacy, fairness and operations before a score is allowed to change how customers are treated?

A data scientist remains responsible for the validation design, the monitoring and the failure cases. The useful outcome of this workshop is that you can tell a careful analysis from a score that has not earned the decision yet.

::: callout-note
## Optional prompt

"What questions should I ask a data scientist before deploying this churn model?"

Compare the answer with the seven questions above.
:::

## Save the test scores

The next workshop decides whom to contact. It needs each test customer's score. It does not need the fitted model.

The cell below writes the combined model's test scores. Those customers were not used to fit the model. The same function accepts `tabular` or `text_only` if you want that model's scores instead.

`SCORE_FILE` is the path of the file this cell writes, folder and file name together. The example is `data/processed/telco/test_probabilities_with_text.csv`. For another dataset, set `SCORE_FILE` to another path, for example `data/processed/mydata/scores.csv`. In the next workshop, set `SCORE_FILE` to that same path. The scores are loaded from the path written there. They are not loaded from `data/processed/telco/` unless that is the path you set.

The file has one row per test customer.

- `customer_id` is the identifier, whatever that column was called in the source table.
- `p0` is the test score.
- `observed_outcome` is 1 when the target equals the positive label, and 0 otherwise.

In [ ]:
SCORE_FILE = "data/processed/telco/test_probabilities_with_text.csv"
workshop.save_scores(combined, SCORE_FILE)

## Troubleshooting

Run the cells from the top if you change a setting. Later cells remember the earlier results until you rerun them.

**The command cannot find the data, or it mentions `_quarto.yml`.** The terminal or the notebook is not in the project folder. That folder contains `_quarto.yml`, `lessons/` and `data/`. Start the terminal there, or rerun the settings cell, which moves up one folder when it cannot see the customer file.

**`ModuleNotFoundError` for pandas, sklearn or matplotlib.** The environment is missing a package or is not the one this document uses. Activate `.venv`, run `pip install -r requirements.txt`, and select the kernel **Python (data-extraction-lesson)**. If that kernel is missing, rerun the `ipykernel install` command in Stage 2.

**A message says a CSV file was not found.** The settings cell names the two files. The example uses `data/raw/telco/telco_customers.csv` and `data/raw/telco/customer_communications.csv`. Replace those two paths when you use another table and another text file of the same kind. The public source file is `data/raw/telco/WA_Fn-UseC_-Telco-Customer-Churn.csv`.

**A message mentions the identifier.** The match between the tabular file and the text file has failed, or an identifier was about to be used as a predictor. Read the full message. Do not delete the identifier from the file; the models need it as a match key and refuse it as a predictor.

**`LOAD` says no saved text features were found, or that the saved file does not match.** `LOAD` is doing what it should. Set `TEXT_MODE` to `"RUN"` to compute the step now, or to `"SAVE"` to compute it and write a file that matches the current seed, split, text file and value of `N_TEXT_COMPONENTS`. Then rerun from the settings cell. Changing *k* requires a new run. An old file for a different *k* is not reused.

**A later cell says a name is not defined.** An earlier cell did not run. Run from the settings cell downward.

**The next workshop cannot find the score file.** Rerun the save cell. Then set `SCORE_FILE` in the next workshop to the same folder and file name, for example `data/processed/mydata/scores.csv`.

**An error message is unclear.**

::: callout-note
## Optional prompt

"Explain this error message and suggest the smallest safe correction."

Paste the error, not a customer extract. Apply the smallest change that the message itself describes, then rerun the affected cell.
:::

## Run the workshop from the start

1.  Open a terminal in the project folder and activate `.venv`.
2.  Open `lessons/02e-churn-text-ml.qmd` and select the kernel **Python (data-extraction-lesson)**.
3.  Run every cell from top to bottom. Leave the settings at the printed defaults on the first pass.
4.  Read each table or chart, then the paragraph under it. The sentences produced by the `explain_` commands use the numbers just calculated.
5.  At the end, read the seven questions in Stage 8 against the final comparison table.
6.  The last modelling cell writes the combined model's test scores to `SCORE_FILE`. The next workshop loads them when its `SCORE_FILE` is that same path. For another dataset, change both, for example to `data/processed/mydata/scores.csv`.

To produce the HTML document instead of running the cells by hand:

``` bash
source .venv/bin/activate
quarto render lessons/02e-churn-text-ml.qmd
```

Run that command from the project folder.